# Lineup xwOBA Analyzer — AL East (2024)

This tool analyzes how each team's 2024 lineup performed against right-handed or left-handed pitching using xwOBA,  a Statcast metric that measures the expected offensive value of a batted ball based on exit velocity and launch angle, removing luck and defensive factors from the equation..

Select a team from the AL East division and a pitcher handedness to generate a breakdown of each batter's xwOBA split for the 2024 season.

**Teams available:** Yankees · Red Sox · Orioles · Blue Jays · Rays  
**Data source:** MLB Statcast via pybaseball  
**Season:** 2024

In [ ]:
!pip install pybaseball pandas

import warnings
warnings.filterwarnings("ignore")

from IPython.display import display, HTML
from pybaseball import statcast_batter, cache

cache.enable()

LINEUPS_2024 = {
    "yankees": [
        ("Juan Soto",665742),("Aaron Judge",592450),("Gleyber Torres",650402),
        ("Jazz Chisholm",665862),("Anthony Rizzo",519203),("Austin Wells",669224),
        ("Alex Verdugo",657077),("Oswaldo Cabrera",665828),("Anthony Volpe",694497)
    ],
    "red sox": [
        ("Rafael Devers",646240),("Jarren Duran",680776),("Tyler O'Neill",641933),
        ("Triston Casas",671213),("Wilyer Abreu",677800),("Ceddanne Rafaela",678882),
        ("Connor Wong",657136),("David Hamilton",666152),("Rob Refsnyder",608701)
    ],
    "orioles": [
        ("Gunnar Henderson",683002),("Adley Rutschman",668939),("Anthony Santander",623993),
        ("Ryan Mountcastle",663624),("Jordan Westburg",676059),("Cedric Mullins",656775),
        ("Colton Cowser",681297),("Ramon Urias",642715),("Jorge Mateo",621035)
    ],
    "blue jays": [
        ("Vladimir Guerrero",665489),("Bo Bichette",666182),("George Springer",543807),
        ("Davis Schneider",676914),("Daulton Varsho",662139),("Alejandro Kirk",672386),
        ("Kevin Kiermaier",595978),("Ernie Clement",676391),("Isiah Kiner-Falefa",643396)
    ],
    "rays": [
        ("Yandy Diaz",650490),("Brandon Lowe",664040),("Randy Arozarena",668227),
        ("Isaac Paredes",670623),("Josh Lowe",666139),("Harold Ramirez",623912),
        ("Jose Siri",642350),("Jose Caballero",676609),("Rene Pinto",650907)
    ],
}

def get_xwoba(player_id, handedness):
    try:
        data = statcast_batter("2024-03-01", "2024-11-30", player_id)
    except Exception:
        return None
    if data is None or data.empty:
        return None
    vs_hand = data[data["p_throws"] == handedness]
    if vs_hand.empty:
        return None
    values = vs_hand["estimated_woba_using_speedangle"].dropna()
    if values.empty:
        return None
    return round(float(values.mean()), 3)

def xwoba_color(val):
    if val is None:  return "#888888"
    if val >= 0.360: return "#4a90d9"
    if val >= 0.290: return "#2e8b57"
    return                  "#c8102e"

def xwoba_grade(val):
    if val is None:  return "N/A"
    if val >= 0.360: return "Elite"
    if val >= 0.290: return "Average"
    return                  "Below Average"

team_name  = input("Team (Yankees / Red Sox / Orioles / Blue Jays / Rays): ").strip()
handedness = ""
while handedness not in ("R", "L"):
    handedness = input("Pitcher hand (R/L): ").strip().upper()

lineup = LINEUPS_2024.get(team_name.lower().strip())
if not lineup:
    print(f"Team '{team_name}' not found. Use: Yankees, Red Sox, Orioles, Blue Jays, or Rays.")
else:
    hand_label = "RHP" if handedness == "R" else "LHP"
    print(f"\nFetching 2024 xwOBA vs {hand_label}...\n")

    rows, xwoba_raw = [], []

    for slot, (name, pid) in enumerate(lineup, 1):
        xwoba = get_xwoba(pid, handedness)
        if xwoba:
            xwoba_raw.append(xwoba)
        rows.append({"slot": slot, "name": name, "xwoba": xwoba})

    html_rows = ""
    for r in rows:
        val       = r["xwoba"]
        color     = xwoba_color(val)
        xwoba_str = f"{val:.3f}" if val else "N/A"
        grade     = xwoba_grade(val)
        html_rows += (
            "<tr>"
            f"<td class='c'>{r['slot']}</td>"
            f"<td>{r['name']}</td>"
            f"<td class='c num' style='color:{color}'>{xwoba_str}</td>"
            f"<td class='c grd' style='color:{color}'>{grade}</td>"
            "</tr>"
        )

    summary = ""
    if xwoba_raw:
        avg       = sum(xwoba_raw) / len(xwoba_raw)
        avg_color = xwoba_color(avg)
        summary = (
            "<tr class='sum'>"
            "<td colspan='2' class='r label'>Lineup Average</td>"
            f"<td class='c num' style='color:{avg_color}'>{avg:.3f}</td>"
            f"<td class='c grd' style='color:{avg_color}'>{xwoba_grade(avg)}</td>"
            "</tr>"
            "<tr class='sum'>"
            "<td colspan='2' class='r label'>MLB Average</td>"
            "<td class='c num' style='color:#888'>0.315</td>"
            "<td class='c grd' style='color:#888'>Benchmark</td>"
            "</tr>"
        )

    legend = (
        "<div style='display:flex;gap:28px;margin-top:14px;font-family:monospace;font-size:12px;color:#aaa'>"
        "<span><span style='color:#4a90d9;font-weight:700'>&#9632;</span> Elite (&ge; .360)</span>"
        "<span><span style='color:#2e8b57;font-weight:700'>&#9632;</span> Average (.290 &ndash; .359)</span>"
        "<span><span style='color:#c8102e;font-weight:700'>&#9632;</span> Below Average (&lt; .290)</span>"
        "</div>"
    )

    html = (
        "<style>"
        "table.xw{border-collapse:collapse;width:540px;font-family:monospace;font-size:14px}"
        "table.xw th{background:#0f1117;color:#ddd;padding:11px 18px;letter-spacing:1px;font-size:12px;text-transform:uppercase}"
        "table.xw td{padding:9px 18px;border-bottom:1px solid #222;background:#161b22;color:#ddd}"
        "table.xw tr:hover td{background:#1c2128}"
        "table.xw .c{text-align:center}"
        "table.xw .r{text-align:right}"
        "table.xw .num{font-weight:700;font-size:15px}"
        "table.xw .grd{font-weight:600;font-size:13px}"
        "table.xw .sum td{background:#0f1117;border-top:2px solid #333}"
        "table.xw .label{color:#aaa;font-size:12px}"
        "</style>"
        f"<h3 style='font-family:monospace;color:#ccc;margin-bottom:6px'>"
        f"{team_name.title()} vs {hand_label} &mdash; 2024</h3>"
        "<table class='xw'>"
        f"<thead><tr><th>#</th><th>Name</th><th>xwOBA vs {hand_label}</th><th>Grade</th></tr></thead>"
        f"<tbody>{html_rows}{summary}</tbody>"
        "</table>"
        f"{legend}"
    )

    display(HTML(html))

Team (Yankees / Red Sox / Orioles / Blue Jays / Rays): blue jays
Pitcher hand (R/L): r

Fetching 2024 xwOBA vs RHP...

Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
